In [3]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["ASTRA_DB_API_ENDPOINT"] = os.getenv("ASTRA_DB_API_ENDPOINT")
os.environ["ASTRA_DB_APPLICATION_TOKEN"] = os.getenv("ASTRA_DB_APPLICATION_TOKEN")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small",dimensions=1024)
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7f5c41b23250>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7f5c41b23b10>, model='text-embedding-3-small', dimensions=1024, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base='https://models.inference.ai.azure.com', openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [13]:
from langchain_astradb import AstraDBVectorStore
vectorstore = AstraDBVectorStore(
    collection_name="langchain_astradb",
    embedding=embeddings
)
vectorstore

In [14]:
## sample code to add documents to the vectorstore
from langchain_core.documents import Document
from langchain_core.documents import Document
sample_documents=[
    Document(
        page_content=""" The Great Wall of China is a series of fortifications made of stone, brick, tamped earth, wood, and other materials. It was built along the northern borders of China to protect against invasions and raids by nomadic groups. The wall stretches over 13,000 miles and is considered one of the greatest architectural feats in history.""",
        metadata={"source": "Wikipedia", "topic": "Great Wall of China", "page": 1}
    ),
    Document(
        page_content =""" The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars in Paris, France. It was named after the engineer Gustave Eiffel, whose company designed and built the tower. The Eiffel Tower is one of the most recognizable structures in the world and is a global cultural icon of France.""",
        metadata={"source": "Wikipedia", "topic": "Eiffel Tower", "page": 1}
    ),
    Document(
        page_content =""" The Pyramids of Giza are ancient pyramid-shaped masonry structures located in Giza, Egypt. They were built as tombs for the pharaohs and their consorts during the Old Kingdom period. The Great Pyramid of Giza is the largest and most famous of the three pyramids and is one of the Seven Wonders of the Ancient World.""",
        metadata={"source": "Wikipedia", "topic": "Pyramids of Giza", "page": 1}
    ),
    Document(
        page_content =""" The Taj Mahal is an ivory-white marble mausoleum located in Agra, India. It was commissioned by the Mughal emperor Shah Jahan in memory of his favorite wife, Mumtaz Mahal. The Taj Mahal is widely recognized as a symbol of love and is one of the most famous buildings in the world.""",
        metadata={"source": "Wikipedia", "topic": "Taj Mahal", "page": 1}   
    ),
    Document(
        page_content =""" The Statue of Liberty is a colossal neoclassical sculpture on Liberty Island in New York Harbor, USA. It was a gift from the people of France to the United States and was designed by French sculptor Frédéric Auguste Bartholdi. The statue is a symbol of freedom and democracy and is one of the most iconic landmarks in the world.""",
        metadata={"source": "Wikipedia", "topic": "Statue of Liberty", "page": 1}
    )
]

In [ ]:
# add documents to the vectorstore
#vectorstore.add_documents(sample_documents)


['285f9946509e41cf85488f9ee44dbdf4',
 '08122c068e7846588e6c8be169c0d89e',
 'ad5fc0127b294e95b7f9fa5145cf9efa',
 'e7c9e1e138f444649403800501c77954',
 '837c8713ae574b6a86de431f344b1f7e']

In [ ]:
# search for similar documents in the vectorstore
vectorstore.similarity_search("What is the Great Wall of China?", k=2)

[Document(id='285f9946509e41cf85488f9ee44dbdf4', metadata={'source': 'Wikipedia', 'topic': 'Great Wall of China', 'page': 1}, page_content=' The Great Wall of China is a series of fortifications made of stone, brick, tamped earth, wood, and other materials. It was built along the northern borders of China to protect against invasions and raids by nomadic groups. The wall stretches over 13,000 miles and is considered one of the greatest architectural feats in history.'),
 Document(id='ad5fc0127b294e95b7f9fa5145cf9efa', metadata={'source': 'Wikipedia', 'topic': 'Pyramids of Giza', 'page': 1}, page_content=' The Pyramids of Giza are ancient pyramid-shaped masonry structures located in Giza, Egypt. They were built as tombs for the pharaohs and their consorts during the Old Kingdom period. The Great Pyramid of Giza is the largest and most famous of the three pyramids and is one of the Seven Wonders of the Ancient World.')]

In [20]:
# search with more filters
result1 = vectorstore.similarity_search(
    "What is located in Champ de Mars in Paris", 
    k=2, 
    filter={"topic": "Eiffel Tower"}
)
print(result1[0].page_content)  # should return the document about the Eiffel Tower
print(result1[0].metadata)  # should return the metadata of the document about the Eiffel Tower

 The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars in Paris, France. It was named after the engineer Gustave Eiffel, whose company designed and built the tower. The Eiffel Tower is one of the most recognizable structures in the world and is a global cultural icon of France.
{'source': 'Wikipedia', 'topic': 'Eiffel Tower', 'page': 1}


### Retriever
The vector store into a retriever, for easier usage in your chains.
Transform the vector store into a retriever and invoke it with a simple query + metadata filter
- search_type- similarity, similarity_score_threshold or mmr


In [21]:
# as a retriever 
retriever = vectorstore.as_retriever(
    search_type = "similarity_score_threshold",
    search_kwargs = {"score_threshold": 0.75}
)

In [23]:
retrieved_docs = retriever.invoke("Tell me about the Great Wall of China")
print(retrieved_docs[0].page_content) 
print(retrieved_docs[0].metadata)

 The Great Wall of China is a series of fortifications made of stone, brick, tamped earth, wood, and other materials. It was built along the northern borders of China to protect against invasions and raids by nomadic groups. The wall stretches over 13,000 miles and is considered one of the greatest architectural feats in history.
{'source': 'Wikipedia', 'topic': 'Great Wall of China', 'page': 1}
